In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import timedelta

# Set visualization style
plt.style.use('ggplot')
sns.set_context("notebook", font_scale=1.2)

# Load the datasets
try:
    filter_flow_df = pd.read_parquet('filterflow_data.parquet')
    error_df = pd.read_parquet('error_data.parquet')
    
    # Convert timestamps to datetime if they aren't already
    filter_flow_df['timestamp'] = pd.to_datetime(filter_flow_df['timestamp'])
    error_df['timestamp'] = pd.to_datetime(error_df['timestamp'])
    
    # Get unique printers
    printers = sorted(filter_flow_df['printer'].unique())
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("This notebook assumes the data files are in the current directory.")

In [ ]:
def create_printer_comparison():
    """Create visualizations comparing all printers"""
    
    # Set up the figure
    plt.figure(figsize=(20, 15))
    
    # 1. Data Collection Density Heatmap
    plt.subplot(2, 2, 1)
    
    # Prepare data for heatmap
    printer_colors = []
    for printer in printers:
        printer_data = filter_flow_df[filter_flow_df['printer'] == printer]
        for color in printer_data['color'].unique():
            color_count = len(printer_data[printer_data['color'] == color])
            printer_colors.append({
                'printer': printer,
                'color': color,
                'count': color_count
            })
    
    density_df = pd.DataFrame(printer_colors)
    density_pivot = density_df.pivot_table(index='printer', columns='color', values='count', fill_value=0)
    
    # Create heatmap - fixing the format issue
    sns.heatmap(density_pivot, annot=True, fmt=".0f", cmap="YlGnBu")
    plt.title('Data Collection Density by Printer and Color')
    plt.ylabel('Printer')
    
    # 2. Error Frequency by Printer
    plt.subplot(2, 2, 2)
    
    # Count errors by printer and color
    error_counts = error_df.groupby('printer').size().reset_index(name='error_count')
    sns.barplot(x='printer', y='error_count', data=error_counts)
    plt.title('Error Frequency by Printer')
    plt.ylabel('Number of Errors')
    plt.xlabel('Printer')
    plt.xticks(rotation=0)
    
    # Add color breakdown as text annotations
    for printer in error_df['printer'].unique():
        color_counts = error_df[error_df['printer'] == printer].groupby('color').size()
        y_pos = error_counts[error_counts['printer'] == printer]['error_count'].values[0]
        plt.text(list(error_counts['printer']).index(printer), y_pos + 1, 
                 '\n'.join([f"{c}: {n}" for c, n in color_counts.items()]),
                 ha='center', va='bottom', fontsize=8)
    
    # 3. Average Filter Flow by Printer
    plt.subplot(2, 2, 3)
    
    avg_flow = filter_flow_df.groupby('printer')['filter_flow'].mean().reset_index()
    sns.barplot(x='printer', y='filter_flow', data=avg_flow)
    plt.title('Average Filter Flow by Printer')
    plt.ylabel('Average Filter Flow')
    plt.xlabel('Printer')
    plt.xticks(rotation=0)
    
    # 4. Data Volume Timeline
    plt.subplot(2, 2, 4)
    
    # Count data points by month
    filter_flow_df['yearmonth'] = filter_flow_df['timestamp'].dt.strftime('%Y-%m')
    monthly_counts = filter_flow_df.groupby('yearmonth').size().reset_index(name='count')
    monthly_counts['date'] = pd.to_datetime(monthly_counts['yearmonth'] + '-01')
    monthly_counts = monthly_counts.sort_values('date')
    
    plt.plot(monthly_counts['date'], monthly_counts['count'], marker='o', linewidth=2)
    plt.title('Monthly Data Volume')
    plt.ylabel('Number of Data Points')
    plt.xlabel('Month')
    
    # Format x-axis dates
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(plt.gca().xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.suptitle('Printer Comparison Dashboard', fontsize=20, y=1.02)
    plt.show()

# Call the comparison function
create_printer_comparison()

In [ ]:
def cluster_printers_simplified():
    """Group printers based on their characteristics using a simpler approach"""
    
    # Create features for each printer
    features = []
    
    for printer in printers:
        printer_data = filter_flow_df[filter_flow_df['printer'] == printer]
        printer_errors = error_df[error_df['printer'] == printer]
        
        if len(printer_data) < 10:  # Skip printers with minimal data
            continue
            
        # Calculate key metrics
        avg_flow = printer_data['filter_flow'].mean()
        data_points = len(printer_data)
        error_count = len(printer_errors)
        
        # Calculate collection frequency (points per day)
        date_range = (printer_data['timestamp'].max() - printer_data['timestamp'].min()).days
        if date_range > 0:
            points_per_day = data_points / date_range
        else:
            points_per_day = 0
            
        # Calculate error rate (per 100 days)
        if date_range > 0:
            error_rate = error_count / date_range * 100
        else:
            error_rate = 0
        
        # Count colors
        unique_colors = len(printer_data['color'].unique())
        
        features.append({
            'printer': printer,
            'avg_flow': avg_flow,
            'points_per_day': points_per_day,
            'error_rate': error_rate,
            'unique_colors': unique_colors
        })
    
    # Convert to DataFrame
    feature_df = pd.DataFrame(features)
    
    # Create a visualization of these features
    if len(feature_df) > 1:  # Need at least 2 printers to compare
        plt.figure(figsize=(15, 10))
        
        # 1. Points per day
        plt.subplot(2, 2, 1)
        sns.barplot(x='printer', y='points_per_day', data=feature_df)
        plt.title('Data Collection Frequency')
        plt.ylabel('Data Points per Day')
        plt.xlabel('Printer')
        
        # 2. Average filter flow
        plt.subplot(2, 2, 2)
        sns.barplot(x='printer', y='avg_flow', data=feature_df)
        plt.title('Average Filter Flow')
        plt.ylabel('Flow Value')
        plt.xlabel('Printer')
        
        # 3. Error rate
        plt.subplot(2, 2, 3)
        sns.barplot(x='printer', y='error_rate', data=feature_df)
        plt.title('Error Rate (per 100 days)')
        plt.ylabel('Errors per 100 Days')
        plt.xlabel('Printer')
        
        # 4. Unique colors
        plt.subplot(2, 2, 4)
        sns.barplot(x='printer', y='unique_colors', data=feature_df)
        plt.title('Number of Colors Tracked')
        plt.ylabel('Unique Colors')
        plt.xlabel('Printer')
        
        plt.tight_layout()
        plt.suptitle('Printer Key Metrics Comparison', fontsize=16, y=1.02)
        plt.show()
        
        # Group printers with similar characteristics
        if len(feature_df) >= 3:  # Need at least 3 printers for meaningful clustering
            # Prepare data for clustering
            X = feature_df.drop('printer', axis=1)
            
            # Normalize features
            from sklearn.preprocessing import StandardScaler
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
            
            # Determine optimal number of clusters
            from sklearn.cluster import KMeans
            from sklearn.metrics import silhouette_score
            
            # Try different numbers of clusters (2 to min(5, n_printers-1))
            max_clusters = min(5, len(feature_df)-1)
            if max_clusters >= 2:
                silhouette_scores = []
                for n_clusters in range(2, max_clusters+1):
                    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
                    cluster_labels = kmeans.fit_predict(X_scaled)
                    silhouette_avg = silhouette_score(X_scaled, cluster_labels)
                    silhouette_scores.append((n_clusters, silhouette_avg))
                
                # Find optimal number of clusters
                optimal_clusters = max(silhouette_scores, key=lambda x: x[1])[0]
                
                # Apply clustering with optimal number
                kmeans = KMeans(n_clusters=optimal_clusters, random_state=42)
                feature_df['cluster'] = kmeans.fit_predict(X_scaled)
                
                # Display results
                print(f"\nPrinters grouped into {optimal_clusters} clusters:")
                for cluster in range(optimal_clusters):
                    printers_in_cluster = feature_df[feature_df['cluster'] == cluster]['printer'].tolist()
                    print(f"Cluster {cluster}: Printers {printers_in_cluster}")
                
                # Visualize clusters
                plt.figure(figsize=(10, 6))
                scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], 
                                      c=feature_df['cluster'], cmap='viridis', 
                                      s=100, alpha=0.8)
                for i, printer in enumerate(feature_df['printer']):
                    plt.annotate(f"P{printer}", (X_scaled[i, 0], X_scaled[i, 1]), 
                                 xytext=(5, 5), textcoords='offset points')
                
                plt.title('Printer Clusters')
                plt.xlabel('Feature 1 (Standardized)')
                plt.ylabel('Feature 2 (Standardized)')
                plt.colorbar(scatter, label='Cluster')
                plt.show()
                
                return feature_df
    
    return feature_df

# Run the clustering
printer_features = cluster_printers_simplified()

In [ ]:
def compare_filter_flow_trends():
    """Compare filter flow trends across printers for specific colors"""
    
    # Choose a specific color to compare (e.g., Black since it's most common)
    for color in ['K', 'C']:  # Compare black and cyan
        # Set up the figure
        plt.figure(figsize=(16, 10))
        plt.title(f'Filter Flow Trends for {color} Color Across Printers', fontsize=16)
        
        # Find printers with this color
        printers_with_color = filter_flow_df[filter_flow_df['color'] == color]['printer'].unique()
        
        # Group data by month
        filter_flow_df['month_year'] = filter_flow_df['timestamp'].dt.to_period('M')
        
        # Plot monthly average filter flow for each printer
        for printer in printers_with_color:
            printer_color_data = filter_flow_df[(filter_flow_df['printer'] == printer) & 
                                             (filter_flow_df['color'] == color)]
            
            if len(printer_color_data) > 10:  # Only plot if we have enough data
                monthly_avg = printer_color_data.groupby('month_year')['filter_flow'].mean().reset_index()
                monthly_avg['month_date'] = monthly_avg['month_year'].dt.to_timestamp()
                
                plt.plot(monthly_avg['month_date'], monthly_avg['filter_flow'], 
                         marker='o', linewidth=2, label=f'Printer {printer}')
        
        plt.ylabel('Average Filter Flow')
        plt.xlabel('Month')
        plt.grid(True, alpha=0.3)
        plt.legend()
        
        # Format x-axis dates
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        plt.setp(plt.gca().xaxis.get_majorticklabels(), rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()

# Compare filter flow trends
compare_filter_flow_trends()

In [ ]:
def create_group_profiles(printer_clusters):
    """Create profiles for each printer group identified in clustering"""
    
    for cluster_id in printer_clusters['cluster'].unique():
        printers_in_cluster = printer_clusters[printer_clusters['cluster'] == cluster_id].index.tolist()
        
        print(f"\n{'='*50}")
        print(f"PROFILE FOR PRINTER GROUP {cluster_id}")
        print(f"{'='*50}")
        print(f"Printers in this group: {printers_in_cluster}")
        
        # Filter data for this cluster
        cluster_data = filter_flow_df[filter_flow_df['printer'].isin(printers_in_cluster)]
        cluster_errors = error_df[error_df['printer'].isin(printers_in_cluster)]
        
        # Summarize key characteristics
        print("\nDATA COLLECTION CHARACTERISTICS:")
        print(f"- Total data points: {len(cluster_data):,}")
        print(f"- Date range: {cluster_data['timestamp'].min().date()} to {cluster_data['timestamp'].max().date()}")
        print(f"- Colors tracked: {', '.join(cluster_data['color'].unique())}")
        
        # Calculate average collection interval
        cluster_data = cluster_data.sort_values(['printer', 'color', 'timestamp'])
        cluster_data['prev_timestamp'] = cluster_data.groupby(['printer', 'color'])['timestamp'].shift(1)
        cluster_data['interval'] = (cluster_data['timestamp'] - cluster_data['prev_timestamp']).dt.total_seconds() / 60
        valid_intervals = cluster_data['interval'].dropna()
        valid_intervals = valid_intervals[valid_intervals < 10080]
        print(f"- Average collection interval: {valid_intervals.mean():.2f} minutes")
        
        # Error patterns
        print("\nERROR PATTERNS:")
        print(f"- Total errors: {len(cluster_errors)}")
        
        # Top error colors
        if not cluster_errors.empty:
            color_errors = cluster_errors.groupby('color').size().reset_index(name='count')
            color_errors = color_errors.sort_values('count', ascending=False)
            print("- Error distribution by color:")
            for _, row in color_errors.iterrows():
                print(f"  * {row['color']}: {row['count']} errors ({row['count']/len(cluster_errors)*100:.1f}%)")
        
        # Calculate error rates
        span_days = (cluster_data['timestamp'].max() - cluster_data['timestamp'].min()).days
        print(f"- Average errors per month: {len(cluster_errors)/(span_days/30):.2f}")
        
        # Filter flow characteristics
        print("\nFILTER FLOW CHARACTERISTICS:")
        print(f"- Normal operating range: {cluster_data['filter_flow'].quantile(0.75):.2f} to {cluster_data['filter_flow'].quantile(0.9):.2f}")
        
        print("\n")

# Generate profiles for printer groups
if 'printer_clusters' in locals():
    create_group_profiles(printer_clusters)